<a href="https://colab.research.google.com/github/op1154/MBA_Agents/blob/main/MBA_Aula2_Langchan_3Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_openai
!pip install langchain_core
!pip install pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [14]:
from pathlib import Path
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from pydantic import Field, BaseModel
from google.colab import userdata

from openai import OpenAI

## Se eu quiser ver como o processo acontece​

from langchain_core.globals import set_debug

set_debug(False)

api_key = userdata.get('OPENAI_API_KEY')


#Classes de saida

class QualProdutoIndicado(BaseModel):

    modelo_indicado: str = Field("Qual produto é mais indicado para o ambiente?")

    motivo: str = Field("Motivo da indicação do produto para o ambiente.")

    ambiente: str = Field("O ambiente para o qual o produto é indicado.") # Added this line

class ProdutosCrossSell(BaseModel):

    produto_Crossindicado: str = Field("Qual produto pode ser indicado como complementar para o ambiente?")

    motivo_Crosssell: str = Field("Motivo da indicação deste novo produto para o ambiente.")

    ambiente_Crosssell: str = Field("Qual o ambiente escolhido.")

parseador = JsonOutputParser(pydantic_object=QualProdutoIndicado)

parseador_crosssell = JsonOutputParser(pydantic_object=ProdutosCrossSell)


#Modelo de Prompts

prompt_produto = PromptTemplate(

    template="""​

    Sugira um modelo de produto dado o meu interesse por {interesse}.​

    {formato_de_saida}​

    """,

    input_variables=["interesse"],

    partial_variables={"formato_de_saida": parseador.get_format_instructions()}

)

modelo_de_produtoCross = PromptTemplate(

    template="""Sugira produtos complementares para o ambiente de {ambiente}.​

    {formato_de_saida}""",

    input_variables=["modelo_indicado", "ambiente"],

    partial_variables={"formato_de_saida": parseador_crosssell.get_format_instructions()}

)

modelo_de_produtoUPSell = PromptTemplate(

    template="Sugira novos produtos necessários para {modelo_indicado} no ambiente de {ambiente}.",

    input_variables=["modelo_indicado", "ambiente"],

)

modelo = ChatOpenAI(

    model="gpt-5-nano",
    temperature=0.3,
    api_key=api_key

)

cadeia_1 = prompt_produto | modelo | parseador

cadeia_2 = modelo_de_produtoCross | modelo | parseador_crosssell

cadeia_3 = modelo_de_produtoUPSell | modelo | StrOutputParser()


Cadeia = cadeia_1 | cadeia_2 | cadeia_3



indicacao = cadeia_1.invoke({"interesse": "TV", "ambiente": "sala de estar"})

cross_sell = cadeia_2.invoke({

    "modelo_indicado": indicacao["modelo_indicado"],

    "ambiente": indicacao["ambiente"],

})

upsell = cadeia_3.invoke({

    "modelo_indicado": indicacao["modelo_indicado"],

    "ambiente": indicacao["ambiente"], # Changed from indicacao["ambiente_Crosssell"]

})

print("Cadeia inteira: ", Cadeia)

print("Produto indicado:", indicacao["modelo_indicado"])

print("Motivo:", indicacao["motivo"])

print("Cross-sell:", cross_sell)

print("Upsell:", upsell)



Cadeia inteira:  first=PromptTemplate(input_variables=['interesse'], input_types={}, partial_variables={'formato_de_saida': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted